← [Overview](00_overview.ipynb)

# Segmentation

Segmentation reduces the number of **timesteps within each period** by merging
adjacent timesteps into fewer segments. It does **not** reduce the number of
periods — that is the job of clustering.

This places segmentation in the **feature-based / resolution-variation** cell of the
Hoffmann (2020) taxonomy.

> **Connection to `contiguous` clustering:**
> [Contiguous clustering](03_agglomerative_clustering.ipynb) and segmentation are the
> **same algorithm** — Ward agglomerative clustering with an adjacency constraint —
> applied at different granularities:
>
> * `contiguous` merges **periods** (rows of the D matrix)
> * segmentation merges **timesteps** within each period
>
> Both are **feature-based** (driven by value similarity), not time-based.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig, SegmentConfig

pio.renderers.default = "notebook_connected"

# --------------------------------------------------------------------------
# Load the shared tiny dataset produced by 01_preprocessing (../tiny.csv).
# --------------------------------------------------------------------------
tiny = pd.read_csv("../tiny.csv", index_col=0, parse_dates=True)
normalized = (tiny - tiny.min()) / (tiny.max() - tiny.min())

# Real dataset
raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
print("tiny:", tiny.shape, "  real:", data.shape)

tiny: (24, 2)   real: (1008, 4)


---

## Mechanism

Constrained agglomerative merging of **adjacent timesteps** within a period:

1. Start with every timestep in its own segment.
2. Find the pair of **adjacent** timesteps whose merge increases within-segment
   variance the least (Ward criterion).
3. Merge them into one segment (represented by their mean).
4. Repeat until $n_{\text{segments}}$ remain.

The **Ward merge cost** between adjacent segments $A$ and $B$ is:

$$
\Delta(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} \| \bar{x}_A - \bar{x}_B \|^2
$$

The cheapest adjacent merge is selected at each step.

**TSAM configuration for segmentation:**

In [2]:
from tsam import ClusterConfig, SegmentConfig
import tsam

# SegmentConfig reduces the number of timesteps within each period.
# n_segments: how many segments per period after merging.
# representation: how each segment is represented (default 'mean').
cfg_seg = SegmentConfig(n_segments=6, representation="mean")
print(cfg_seg)

# Segmentation is passed separately to tsam.aggregate — it can be combined
# with any clustering method:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=ClusterConfig(method="hierarchical"),
#     segments=SegmentConfig(n_segments=6),
# )

# Segmentation alone (no clustering — n_clusters=1, all periods in one cluster):
# result = tsam.aggregate(
#     df,
#     n_clusters=len(df) // timesteps_per_period,
#     period_duration="1D",
#     segments=SegmentConfig(n_segments=6),
# )

SegmentConfig(n_segments=6, representation='mean')


In [3]:
# Manual segmentation of day_4 (cloudy day) from 4 -> 2 segments
day4_norm = normalized.iloc[16:20]  # timesteps 0-3 of day 4
print("Day 4 normalized values:")
print(day4_norm.to_string())

vals = day4_norm.values  # shape (4, 2)
print("\nWard merge cost for each adjacent pair:")
for i in range(3):
    pair = vals[i:i+2]
    mean_pair = pair.mean(axis=0)
    cost = float(np.sum((pair - mean_pair) ** 2))
    print(f"  merge t{i}+t{i+1}: cost = {cost:.4f}")
print("The minimum-cost adjacent pair is merged first.")

Day 4 normalized values:
                     solar      load
time                                
2020-01-05 00:00:00  0.000  0.428571
2020-01-05 06:00:00  0.125  0.428571
2020-01-05 12:00:00  0.125  0.571429
2020-01-05 18:00:00  0.000  0.571429

Ward merge cost for each adjacent pair:
  merge t0+t1: cost = 0.0078
  merge t1+t2: cost = 0.0102
  merge t2+t3: cost = 0.0078
The minimum-cost adjacent pair is merged first.


In [4]:
# tsam: clustering + segmentation on tiny data
result_seg_tiny = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    segments=SegmentConfig(n_segments=2),
)
print("Segmented tiny result:")
print("  n_clusters:", result_seg_tiny.n_clusters)
print("  n_segments:", result_seg_tiny.n_segments)
print("\nSegmented cluster representatives (each period has 2 segments):")
result_seg_tiny.cluster_representatives

Segmented tiny result:
  n_clusters: 3
  n_segments: 2

Segmented cluster representatives (each period has 2 segments):


solar      load
  Segment Step Segment Duration                    
0 0            2                 0.488095  6.180000
  1            2                 0.488095  7.240000
1 0            3                 4.555556  3.353333
  1            1                 0.000000  5.120000
2 0            3                 1.626984  4.766667
  1            1                 0.000000  6.180000

In [5]:
# tsam: segmentation on real data (24h -> 6 segments per period)
result_seg = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    segments=SegmentConfig(n_segments=6),
)
print("Segmented (real) — n_segments:", result_seg.n_segments)
print("Weighted RMSE:", round(result_seg.accuracy.weighted_rmse, 4))

result_seg.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="Segmented (6 segments/day) — Load duration curve vs original",
)

Segmented (real) — n_segments: 6


Weighted RMSE: 0.1271


In [6]:
# Effect of n_segments on accuracy
rows = []
for ns in [2, 4, 6, 12, 24]:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D",
        cluster=ClusterConfig(method="hierarchical"),
        segments=SegmentConfig(n_segments=ns),
    )
    rows.append({"n_segments": ns, "weighted_rmse": round(r.accuracy.weighted_rmse, 4)})

df_seg = pd.DataFrame(rows)
print("n_segments vs accuracy (k=6 hierarchical, 6-week real data):")
df_seg

n_segments vs accuracy (k=6 hierarchical, 6-week real data):


,n_segments,weighted_rmse
0,2,0.1491
1,4,0.1310
2,6,0.1271
3,12,0.1237
4,24,0.1235


In [7]:
px.line(
    df_seg, x="n_segments", y="weighted_rmse",
    markers=True,
    title="Accuracy vs number of segments per period (k=6)",
    labels={"weighted_rmse": "Weighted RMSE", "n_segments": "Segments per period"},
)

---

**See also:**
* [Contiguous clustering](03_agglomerative_clustering.ipynb) — same algorithm at the period level
* [Representation](06_representation.ipynb) — controlling how each segment is represented